# 06 - Word Embedding (W2V)

## Setup

In [30]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

from gensim.corpora import Dictionary
from gensim.models import word2vec

from sklearn.manifold import TSNE as tsne

OUTPUT_DIR = 'tables'

In [31]:
import warnings; warnings.filterwarnings('ignore', category=FutureWarning)
import gensim; gensim.__version__

'4.4.0'

In [32]:
OHCO = ['film_name', 'chunk_num', 'sent_num', 'token_num']
bags = dict(
    SENTS = OHCO[:3],
    CHUNKS = OHCO[:2],
    FILMS = OHCO[:1]
)

# Import tables
LIB = pd.read_csv('tables/LIB.csv').set_index('film_name')
CORPUS = pd.read_csv('tables/CORPUS.csv').set_index(OHCO)

### Convert to Gensim

In [33]:
docs = CORPUS.groupby(bags['CHUNKS']).term_str.apply(list).tolist()

In [34]:
# inspect first 5 documents of Gensim corpus
for i in range(5):
    print(f"Doc {i}:", docs[i])

Doc 0: ['ah']
Doc 1: ['wah', 'it', 's', 'a', 'gas', 'bomb', 'it', 's', 'an', 'attack']
Doc 2: ['wah']
Doc 3: ['ah', 'wah', 'hey']
Doc 4: ['hold', 'them', 'there', 'you', 'get', 'down', 'on', 'the', 'floor']


In [35]:
dictionary = Dictionary(docs)

### Generate word embeddings with Gensim's module

In [36]:
w2v_params = dict(
    window = 10,
    vector_size = 100,
    sg = 1, # add skip-gram to train on word-context pair individually, given limited data
    min_count = 20,
    workers = 1,
    seed = 42
)

In [37]:
model = word2vec.Word2Vec(docs, **w2v_params)
model.wv.vectors

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


array([[-0.14695631,  0.02533271, -0.01588253, ..., -0.00136856,
        -0.07541825, -0.11001442],
       [ 0.15919504,  0.00302501,  0.14197564, ..., -0.03822015,
        -0.10148197, -0.09972103],
       [-0.09574412,  0.17312099,  0.18210033, ...,  0.04820444,
        -0.14449485, -0.29991788],
       ...,
       [-0.0564752 ,  0.11640438, -0.0023535 , ...,  0.0166597 ,
        -0.07901452, -0.13590705],
       [-0.04240542,  0.07950036,  0.00631212, ..., -0.02395877,
        -0.07505882, -0.10760974],
       [-0.06186857,  0.08860616,  0.00593281, ..., -0.00475474,
        -0.07336484, -0.11415359]], shape=(577, 100), dtype=float32)

In [38]:
VOCAB_W2V = pd.DataFrame(model.wv.vectors, index=model.wv.index_to_key)
VOCAB_W2V.index.name = 'term_str'
VOCAB_W2V

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
term_str,,,,,,,,,,,,,,,,,,,,,
the,-0.146956,0.025333,-0.015883,0.154385,-0.304915,0.344831,-0.083909,0.243847,-0.206692,-0.101077,...,-0.019560,0.313509,0.096871,0.098985,0.101150,-0.012153,-0.004173,-0.001369,-0.075418,-0.110014
you,0.159195,0.003025,0.141976,-0.083872,-0.000778,0.165070,0.133059,0.106482,-0.096522,0.149340,...,-0.062026,0.087579,0.168215,-0.268036,0.105097,-0.071202,0.072089,-0.038220,-0.101482,-0.099721
i,-0.095744,0.173121,0.182100,0.101251,0.013899,-0.005879,0.047531,0.244172,-0.037220,0.226875,...,-0.095561,0.143059,0.314360,-0.415737,0.066268,0.026788,0.065066,0.048204,-0.144495,-0.299918
s,-0.182771,0.229002,-0.110760,0.194948,-0.121372,-0.126508,-0.232620,-0.083815,-0.206123,0.223268,...,0.040593,0.012912,0.068300,0.038977,0.124497,0.280367,-0.000956,-0.235553,-0.328026,-0.042805
to,0.148523,0.144695,-0.010882,0.145847,-0.254098,0.124183,0.015818,0.094736,-0.086314,-0.083848,...,0.022571,0.180348,0.237386,-0.193600,0.007446,0.057996,0.032166,0.037096,0.068112,-0.193112
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
friends,-0.045368,0.069984,0.011366,0.100150,-0.252643,0.200876,-0.037885,0.149814,-0.178569,0.056011,...,-0.166047,0.114743,0.070807,-0.069578,0.023645,-0.001649,0.038183,-0.010527,-0.077606,-0.129995
shut,-0.019380,0.060844,0.003102,0.083030,-0.214565,0.108152,-0.066532,0.099882,-0.182831,0.065906,...,-0.121351,0.145316,0.095366,-0.075659,-0.005192,0.047829,0.049016,-0.021939,-0.042627,-0.140790
clan,-0.056475,0.116404,-0.002353,0.100070,-0.316149,0.266264,-0.029852,0.218061,-0.177987,0.015637,...,-0.169714,0.125737,0.083620,-0.019733,0.024308,-0.019717,0.061087,0.016660,-0.079015,-0.135907


### Visualize with tSNE

In [39]:
PP = 40 # Try 1, 100, etc.

tsne_engine = tsne(
    perplexity=PP, 
    n_components=2, 
    init='random', 
    max_iter=2500, 
    random_state=23
)
TSNE = pd.DataFrame(
    tsne_engine.fit_transform(VOCAB_W2V), 
    columns=['x','y'], 
    index=VOCAB_W2V.index)
TSNE

,x,y
term_str,,
the,23.636288,-0.399716
you,-15.894247,8.804877
i,-18.698149,-1.353485
s,-5.075657,-11.342407
to,-5.650084,10.434741
...,...,...
friends,10.354618,-0.169740
shut,-2.259795,7.057340
clan,17.779058,-2.255396


In [40]:
px.scatter(TSNE.reset_index(), 'x', 'y', 
        text='term_str', 
        hover_name='term_str',  
        height=1000,
        width=1200)\
    .update_traces(
        mode='markers+text', 
        textfont=dict(color='black', size=14, family='Arial'),
        textposition='top center')

## Save Table

In [41]:
VOCAB_W2V.to_csv(f'{OUTPUT_DIR}/VOCAB_W2V.csv')